In [2]:
import sys
import os.path as osp

PROJECT_DIR = '../../'
PROJECT_DIR = osp.abspath(PROJECT_DIR)
print(PROJECT_DIR in sys.path)
if PROJECT_DIR not in sys.path:
    print(f'Adding project directory to the sys.path: {PROJECT_DIR!r}')
    sys.path.insert(1, PROJECT_DIR)

True


In this experiment, we'll solve a problem of recommending a user a specific genre of the films he likes the most. Each of the available 18 genres from the `movies` table. Each arm will be an action of selecting a movie belonging to the respective genre, and then returning a reward. The reward will be zero if the user in question has not rated the movie at all, while otherwise it will range from 0.5 to 1 depending on the rating in 1-5 that the user has given to the movie. The corresponding regret will be the difference between the most optimal scenario and the one produced by the action, so, in this case, that will be the reward for the user's rating of 5. Therefore, the regret here is essentially similified to being 1 - reward.

The choice of the movie by each arm will be random, with each movie weighted according to its popularity.

The initial run of the algorithm takes place at the selected time point, which can be assigned by user or randomly within the dataset time range. Then, after the simulation took place, the algorithm "waits" for the new data, and when either (a) the selected periodicity of online learning is reached or (b) the selected upper boundary of the data required to trigger the re-learning is reached, the algorithm updates the regrets for each arms by running the simulation on the new data.

With the definition of the problem set and the conditions for the bandit outlined, let's develop the bandit and the strategies for it. Here we will analyze all three most popular strategies:

- epsilon-greed strategy;
- UCB strategy (in particular, the method would be based on the UCB1 approach);
- Thompson sampling.

First, let's setup the needed libraries and load the data:

In [4]:
import numpy as np
import pandas as pd
import scipy
from tqdm.notebook import tqdm
import json

In [5]:
df_ratings = pd.read_csv('../../data/ml-1m/ratings.dat',
                         delimiter='::',
                         header=None,
                         names=['UserID','MovieID','Rating','Timestamp'],
                         engine ='python')

In [6]:
df_ratings.head()

,UserID,MovieID,Rating,Timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


In [8]:
df_movies = pd.read_csv('../../data/ml-1m/movies.dat',
                         delimiter='::',
                         header=None,
                         names=['MovieID','Title','Genres'],
                         encoding='latin-1')

/tmp/ipykernel_12551/3129768434.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df_movies = pd.read_csv('../../data/ml-1m/movies.dat',


In [9]:
df_movies.head()

,MovieID,Title,Genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


Let's calculate the weights for the random movie recommendation by each arm:

In [10]:
count_ratings = df_ratings.groupby('MovieID')['Rating'].count()

In [35]:
count_ratings.sort_values(ascending=False)

MovieID
2858    3428
260     2991
1196    2990
1210    2883
480     2672
        ... 
3237       1
763        1
624        1
2563       1
3290       1
Name: Rating, Length: 3706, dtype: int64

In [36]:
movie_popularity = count_ratings.to_dict()

Let's calculate a list of movies for each genre. Worth pointing out, that the movie can belong to the several genres, and therefore appear in the several such lists:

In [26]:
genres_list = np.unique([genre for movie_genres in df_movies['Genres'].str.split('|') for genre in movie_genres])
genres_list

array(['Action', 'Adventure', 'Animation', "Children's", 'Comedy',
       'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
       'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War',
       'Western'], dtype='<U11')

In [33]:
genres_movie_list = {genre : df_movies[df_movies['Genres'].str.contains(genre)][
                     'MovieID'].tolist() for genre in genres_list}

And then, based on these list and overall movie popularity in them, let's calculate the weights of the movies in each genre. For each genre we will also calculate the cumulative sum of these weights for the easy distribution sampling.

The movies with zero popularity (i.e. those that are in the `movies` table, but have zero ratings) will receive the popularity rating of 0.5, as they are also part of their genre's movie distribution, and the fact that nobode rated them can be valuable for the recommendation and our bandit's decision.

In [47]:
genres_movie_popularity_sum = {genre : np.sum([movie_popularity[
    movie] if movie in movie_popularity else 0.5 for movie in genres_movie_list[genre]]) for genre in genres_list}
print(genres_movie_popularity_sum)

{'Action': 257461.0, 'Adventure': 133954.0, 'Animation': 43293, "Children's": 72186.5, 'Comedy': 356598.5, 'Crime': 79546.0, 'Documentary': 7918.5, 'Drama': 354584.0, 'Fantasy': 36301, 'Film-Noir': 18261, 'Horror': 76388.0, 'Musical': 41533.5, 'Mystery': 40179.0, 'Romance': 147529.0, 'Sci-Fi': 157295.0, 'Thriller': 189683.5, 'War': 68528.0, 'Western': 20683.5}


In [48]:
genres_movie_weights = {genre : [(movie_popularity[
    movie] if movie in movie_popularity else 0.5)/genres_movie_popularity_sum[
        genre] for movie in genres_movie_list[genre]] for genre in genres_list}

In [50]:
genres_movie_cumulative_sum = {genre : np.cumsum(genres_movie_weights[genre]) for genre in genres_list}

In [52]:
genres_movie_cumulative_sum['Action'][:10]

array([0.00365104, 0.00404721, 0.00749628, 0.00806336, 0.00868481,
       0.01395163, 0.01481001, 0.01601796, 0.0160199 , 0.01956995])

This will be used to randomly sample the movies for each genre according to their popularity.

Now, let's develop the multi-armed bandit:

In [61]:
# The principle of movie weighted random recommendation
a = np.random.uniform()
print(a, np.sum(genres_movie_cumulative_sum['Action'] < a))

0.686996931754785 309


In [73]:
user_movie_ratings = df_ratings.set_index(['UserID','MovieID'])['Rating'].to_dict()

In [266]:
class MultiArmedBandit:
    def __init__(self, arms_number, arms_names, arms_items, arms_choice_cumulative_sum, user_item_ratings):
        self.arms_number = arms_number # In our case, 18 genres
        self.arms_names = arms_names # Genre names
        self.arms_items = arms_items # Movies by genre
        self.arms_choice_cumulative_sum = arms_choice_cumulative_sum
        self.user_item_ratings = user_item_ratings
        self.round_number = 0

    def play_next_round(self, arm, user_id, verbose: int = 0):
        # This function "plays" a round on the bandit, and returns the regret
        assert arm < self.arms_number
        arm_name = self.arms_names[arm]
        selected_item = np.sum(self.arms_choice_cumulative_sum[arm_name] < np.random.uniform())
        selected_item = self.arms_items[arm_name][selected_item]
        if tuple([user_id, selected_item]) in self.user_item_ratings:
            rating = self.user_item_ratings[tuple([user_id, selected_item])]
            regret = 0.625 - rating/8 # Regret = 1 - reward = 1 - ((rating-1)/(4/0.5) + 0.5)
            # The resulting regret is in [0;0.5] if the movie was watched
        else:
            regret = 1
        if verbose == 1:
            print(f'--Round {self.round_number} arm {arm} {arm_name} selected_item {selected_item} regret {regret}' )
        self.round_number += 1
        return regret

In [267]:
bandit_test = MultiArmedBandit(len(genres_list),
                               genres_list,
                               genres_movie_list,
                               genres_movie_cumulative_sum,
                               user_movie_ratings)

In [268]:
bandit_test.play_next_round(4, 1000, verbose=1)

--Round 0 arm 4 Comedy selected_item 3591 regret 1


1

In [270]:
df_movies[df_movies['MovieID'] == 3591]

,MovieID,Title,Genres
3522,3591,Mr. Mom (1983),Comedy|Drama


Let's now develop the strategies:

In [ ]:
(q*n + r)/(n+1) = q + (r - q)/(n+1)

In [327]:
class UCBStrategy:
    def __init__(self, bandit_object, const_confidence: float = 1.0,
                 epsilon: float = 1e-8, Q_initial = [], rn_initial = []):
        self.strategy_type = 'ucb'
        self.bandit_object = bandit_object # We obviously use the same bandit setup for the same strategy
        self.round_number = 0 # Overall round number for the strategy
        if len(Q_initial) == 0:
            self.Q_arms = np.zeros(self.bandit_object.arms_number)
        else:
            self.Q_arms = Q_initial
        if len(rn_initial) == 0:
            self.round_numbers_arms = np.zeros(self.bandit_object.arms_number) # Round number for the individual arms
        else:
            self.round_numbers_arms = rn_initial
        self.const_confidence = const_confidence
        self.epsilon = epsilon

    def next_step(self, user_id):
        action_step = self.select_new_action()
        regret = self.bandit_object.play_next_round(action_step, user_id)
        self.increment_values(action_step, regret)
        return regret

    def select_new_action(self):
        if self.round_number > 0:
            confidences = np.sqrt(2*np.log(self.round_number)/(self.round_numbers_arms + self.epsilon))
        else:
            confidences = np.zeros(self.bandit_object.arms_number)
        return np.argmax(self.Q_arms + self.const_confidence*confidences)

    def increment_values(self, action_step, regret):
        self.round_number += 1
        self.round_numbers_arms[action_step] += 1
        # (q*n + reward)/(n+1) = q + (reward - q)/(n+1) = q + (1 - regret - q)/(n+1)
        self.Q_arms[action_step] += (1 - regret - self.Q_arms[action_step])/self.round_numbers_arms[action_step]

In [272]:
ucb_strategy_test = UCBStrategy(bandit_test)

In [273]:
ucb_strategy_test.next_step(1000)

1

In [328]:
class EpsilonGreedyStrategy:
    def __init__(self, bandit_object, epsilon_parameter: float = 0.2,
                 epsilon: float = 1e-8, Q_initial = [], rn_initial = []):
        self.strategy_type = 'eg'
        self.bandit_object = bandit_object # We obviously use the same bandit setup for the same strategy
        self.epsilon_parameter = epsilon_parameter
        self.round_number = 0 # Overall round number for the strategy
        if len(Q_initial) == 0:
            self.Q_arms = np.zeros(self.bandit_object.arms_number)
        else:
            self.Q_arms = Q_initial
        if len(rn_initial) == 0:
            self.round_numbers_arms = np.zeros(self.bandit_object.arms_number) # Round number for the individual arms
        else:
            self.round_numbers_arms = rn_initial
        self.epsilon = epsilon

    def next_step(self, user_id):
        action_step = self.select_new_action()
        regret = self.bandit_object.play_next_round(action_step, user_id)
        self.increment_values(action_step, regret)
        return regret

    def select_new_action(self):
        if np.random.uniform() > self.epsilon_parameter:
            return np.argmax(self.Q_arms)
        else:
            return np.random.randint(self.bandit_object.arms_number)

    def increment_values(self, action_step, regret):
        self.round_number += 1
        self.round_numbers_arms[action_step] += 1
        # (q*n + reward)/(n+1) = q + (reward - q)/(n+1) = q + (1 - regret - q)/(n+1)
        self.Q_arms[action_step] += (1 - regret - self.Q_arms[action_step])/self.round_numbers_arms[action_step]

In [311]:
bandit_test_eg = MultiArmedBandit(len(genres_list),
                               genres_list,
                               genres_movie_list,
                               genres_movie_cumulative_sum,
                               user_movie_ratings)
eg_strategy_test = EpsilonGreedyStrategy(bandit_test_eg)

In [312]:
eg_strategy_test.next_step(1000)

1

In [329]:
class ThompsonSamplingStrategy:
    def __init__(self, bandit_object, epsilon: float = 1e-8, Q_initial = [], rn_initial = []):
        self.strategy_type = 'ts'
        self.bandit_object = bandit_object # We obviously use the same bandit setup for the same strategy
        self.round_number = 0 # Overall round number for the strategy
        # For this strategy, we store alpha paraneter here: (ideally we need to rename this variable)
        if len(Q_initial) == 0:
            self.Q_arms = np.ones(self.bandit_object.arms_number)
        else:
            self.Q_arms = Q_initial
            # For this strategy, we store beta paraneter here: (ideally we need to rename this variable)
        if len(rn_initial) == 0:
            self.round_numbers_arms = np.ones(self.bandit_object.arms_number) # Round number for the individual arms
        else:
            self.round_numbers_arms = rn_initial
        self.epsilon = epsilon

    def next_step(self, user_id):
        action_step = self.select_new_action()
        regret = self.bandit_object.play_next_round(action_step, user_id)
        self.increment_values(action_step, regret)
        return regret

    def select_new_action(self):
        return np.argmax(np.random.beta(self.Q_arms, self.round_numbers_arms)) # (alpha, beta)

    def increment_values(self, action_step, regret):
        self.round_number += 1
        # alpha
        self.Q_arms[action_step] += 1 - regret # Increasing with reward, which is in the range [0;1]
        # beta
        self.round_numbers_arms[action_step] += regret # Increasimng with 1 - reward

In [323]:
bandit_test_ts = MultiArmedBandit(len(genres_list),
                               genres_list,
                               genres_movie_list,
                               genres_movie_cumulative_sum,
                               user_movie_ratings)
ts_strategy_test = ThompsonSamplingStrategy(bandit_test_ts)

In [324]:
ts_strategy_test.next_step(1000)

1

And then, let's define a function to run learning at a selected timestamp. The learning is aimed to be run for each particular user.

In [176]:
-np.sort(-np.array([1,3,2]))

array([3, 2, 1])

In [177]:
np.argsort(-np.array([1,3,2]))

array([1, 2, 0])

In [333]:
class OnlineStrategyLearner:
    def __init__(self, data_ratings, arms_number, arms_names, arms_items,
                 timestamp, periodicity_minutes, periodicity_records,
                 bandit_class, strategy_class, user_id, max_number_rounds: int = 1000,
                 max_number_online_rounds: int = 100):
        self.arms_number = arms_number # In our case, 18 genres
        self.arms_names = arms_names # Genre names
        self.arms_items = arms_items # Movies by genre
        self.data_ratings = data_ratings
        self.timestamp = timestamp
        self.periodicity_minutes = periodicity_minutes
        self.periodicity_records = periodicity_records
        self.bandit_class = bandit_class
        self.strategy_class = strategy_class
        self.user_id = user_id
        self.max_number_rounds = max_number_rounds
        self.max_number_online_rounds = max_number_online_rounds
        self.epsilon = 1e-8

    def run_initial_learning(self):
        arms_choice_cumulative_sum, user_item_ratings = self.recalculate_time_dependent_parameters(self.timestamp)
        bandit_object = self.bandit_class(self.arms_number,
                               self.arms_names,
                               self.arms_items,
                               arms_choice_cumulative_sum,
                               user_item_ratings)
        strategy_object = self.strategy_class(bandit_object)
        regrets = []
        for i_round in range(self.max_number_rounds):
            regrets.append(strategy_object.next_step(self.user_id))
        self.Q_initial = strategy_object.Q_arms
        self.rn_initial = strategy_object.round_numbers_arms
        if strategy_object.strategy_type != 'ts':
            top_actions = list(zip(np.argsort(-strategy_object.Q_arms),
                                   self.arms_names[np.argsort(-strategy_object.Q_arms)],
                                   -np.sort(-strategy_object.Q_arms)))
        else:
            alpha_beta_proportion = strategy_object.Q_arms/(strategy_object.round_numbers_arms + self.epsilon)
            top_actions = list(zip(np.argsort(-alpha_beta_proportion),
                                   self.arms_names[np.argsort(-alpha_beta_proportion)],
                                   strategy_object.Q_arms[np.argsort(-alpha_beta_proportion)],
                                   strategy_object.round_numbers_arms[np.argsort(-alpha_beta_proportion)]))
        print(f'--Timestamp {self.timestamp} after {self.max_number_rounds} iterations: sum of regrets {np.sum(regrets)} top actions {top_actions[:5]}')
        return regrets

    def run_online_learning(self, steps_ahead):
        regrets_steps = []
        for step_ahead in range(1,steps_ahead+1):
            new_timestamp = self.timestamp+step_ahead*self.periodicity_minutes
            arms_choice_cumulative_sum, user_item_ratings = self.recalculate_time_dependent_parameters(
                new_timestamp)
            bandit_object = self.bandit_class(self.arms_number,
                               self.arms_names,
                               self.arms_items,
                               arms_choice_cumulative_sum,
                               user_item_ratings)
            strategy_object = self.strategy_class(bandit_object,
                                                  Q_initial=self.Q_initial,
                                                  rn_initial=self.rn_initial)
            regrets = []
            for i_round in range(self.max_number_online_rounds):
                regrets.append(strategy_object.next_step(self.user_id))
            if strategy_object.strategy_type != 'ts':
                top_actions = list(zip(np.argsort(-strategy_object.Q_arms),
                                       self.arms_names[np.argsort(-strategy_object.Q_arms)],
                                       -np.sort(-strategy_object.Q_arms)))
            else:
                alpha_beta_proportion = strategy_object.Q_arms/(strategy_object.round_numbers_arms + self.epsilon)
                top_actions = list(zip(np.argsort(-alpha_beta_proportion),
                                       self.arms_names[np.argsort(-alpha_beta_proportion)],
                                       strategy_object.Q_arms[np.argsort(-alpha_beta_proportion)],
                                       strategy_object.round_numbers_arms[np.argsort(-alpha_beta_proportion)]))
            self.Q_initial = strategy_object.Q_arms
            self.rn_initial = strategy_object.round_numbers_arms
            print(f'--Step {step_ahead} new_timestamp {new_timestamp} after {self.max_number_online_rounds} iterations: sum of regrets {np.sum(regrets)} top actions {top_actions[:5]}')
            regrets_steps.append(regrets)
        return regrets_steps

    def recalculate_time_dependent_parameters(self, timestamp, timestamps_before: int = -1):
        if timestamps_before == -1: # Initial learning
            new_data_ratings = self.data_ratings[self.data_ratings['Timestamp'] <= timestamp]
        else:
            new_data_ratings = self.data_ratings[(self.data_ratings['Timestamp'] > timestamps_before) & (
                self.data_ratings['Timestamp'] <= timestamp)]
        movie_popularity = new_data_ratings.groupby(
            'MovieID')['Rating'].count().to_dict()
        genres_movie_popularity_sum = {genre : np.sum([movie_popularity[
            movie] if movie in movie_popularity else 0.5 for movie in self.arms_items[
                genre]]) for genre in self.arms_names}
        genres_movie_weights = {genre : [(movie_popularity[
            movie] if movie in movie_popularity else 0.5)/genres_movie_popularity_sum[
                genre] for movie in self.arms_items[genre]] for genre in self.arms_names}
        genres_movie_cumulative_sum = {genre : np.cumsum(genres_movie_weights[genre]) for genre in self.arms_names}
        user_movie_ratings = new_data_ratings.set_index([
            'UserID','MovieID'])['Rating'].to_dict()
        return genres_movie_cumulative_sum, user_movie_ratings

## Strategies online learning and review

Let's run the **UCB strategy** for the most active user, to better demonstrate the properties of the strategy:

In [197]:
df_ratings.groupby('UserID')['Rating'].count().sort_values(ascending=False).head()

UserID
4169    2314
1680    1850
4277    1743
1941    1595
1181    1521
Name: Rating, dtype: int64

In [198]:
print(df_ratings['Timestamp'].min(), df_ratings['Timestamp'].max())

956703932 1046454590


In [199]:
start_timestamp = int(df_ratings['Timestamp'].min() + 0.8*(
    df_ratings['Timestamp'].max() - df_ratings['Timestamp'].min()))
start_timestamp

1028504458

In [297]:
online_strategy_user = OnlineStrategyLearner(df_ratings,
                                             len(genres_list),
                                             genres_list,
                                             genres_movie_list,
                                             start_timestamp,
                                             24*60, # Update each day = 24*60 minutes
                                             10000,
                                             MultiArmedBandit,
                                             UCBStrategy,
                                             user_id=4169,
                                             max_number_rounds=10000,
                                             max_number_online_rounds=1000)

In [298]:
initial_regrets = online_strategy_user.run_initial_learning()

--Timestamp 1028504458 after 10000 iterations: sum of regrets 1386.5 top actions [(9, 'Film-Noir', 0.9481069156577034), (16, 'War', 0.8670154185022025), (5, 'Crime', 0.8322091062394597), (7, 'Drama', 0.8259981851179671), (17, 'Western', 0.7931472081218277)]


In [299]:
print(initial_regrets[:25])

[0.25, 0.125, 0.125, 0.0, 0.0, 1, 0.375, 1, 0.0, 0.25, 0.0, 0.25, 0.125, 0.125, 0.25, 1, 1, 0.25, 0.125, 0.0, 1, 0.25, 0.0, 0.125, 0.0]


As we can see, the best actions for this user according to the UCB strategy were determined to be selecting the genres "Film-Noir", which has an average reward of 0.948 (and the corresponding average regret of 0.052; the reward was printed here for more highlighting the better scores for these genres); or then "War", "Crime", "Drama", or "Western". And we can see how it already lays out some picture about the user's preferences!

Let's also run (or rather simulate) online learning for each day of the next week, as we had set the periodicity of online updates as 24*60 minutes = 1 day, so we can achieve this with setting `steps_ahead`=7:

In [300]:
online_regrets = online_strategy_user.run_online_learning(steps_ahead=7)

--Step 1 new_timestamp 1028505898 after 1000 iterations: sum of regrets 53.25 top actions [(9, 'Film-Noir', 0.9478758726374913), (16, 'War', 0.8670154185022025), (5, 'Crime', 0.8322091062394597), (7, 'Drama', 0.8259981851179671), (17, 'Western', 0.7931472081218277)]
--Step 2 new_timestamp 1028507338 after 1000 iterations: sum of regrets 48.75 top actions [(9, 'Film-Noir', 0.9483667976138506), (16, 'War', 0.8670154185022025), (5, 'Crime', 0.8322091062394597), (7, 'Drama', 0.8259981851179671), (17, 'Western', 0.7931472081218277)]
--Step 3 new_timestamp 1028508778 after 1000 iterations: sum of regrets 52.75 top actions [(9, 'Film-Noir', 0.9482649040294893), (16, 'War', 0.8672045951859955), (5, 'Crime', 0.8322091062394597), (7, 'Drama', 0.8259981851179671), (17, 'Western', 0.7931472081218277)]
--Step 4 new_timestamp 1028510218 after 1000 iterations: sum of regrets 58.75 top actions [(9, 'Film-Noir', 0.9479402765185846), (16, 'War', 0.8665099268547541), (5, 'Crime', 0.8322091062394597), (7,

As we can see, the user most likely haven't watched any westerns in the following week, thus the average reward for recommending this genre did not change.

One another thing worth pointing out here is that here we estimate the best genres for the user based on his own ratings history, but for some special scenarios, like the cold start, we can easily replace them with other sources to measure reward, and therefore regret. A good idea would be to find the closest users to this one by their ratings, or, for example, some website matedata, and use their scores as the reward ratings for recommending a certain movie.

Another advantage of such a system is that the compuations are really quick. Compared to some other ecommendation ststem's we've tried, this one is able to compute the best genres for a user in just seconds. Part of this could be attributed to some degree of optimization in-build in the algorithms already while developing, such as avoiding using `pandas` in the loops altogether and using raw data formats more frequently. This is certainly proving to be a useful strategy for a large dataset.

Let's also review what the other strategies determine to be the best genres for this user duting this week, starting from the **epsilon-greedy strategy**.

In [313]:
online_strategy_user_eg = OnlineStrategyLearner(df_ratings,
                                             len(genres_list),
                                             genres_list,
                                             genres_movie_list,
                                             start_timestamp,
                                             24*60, # Update each day = 24*60 minutes
                                             10000,
                                             MultiArmedBandit,
                                             EpsilonGreedyStrategy,
                                             user_id=4169,
                                             max_number_rounds=10000,
                                             max_number_online_rounds=1000)

In [314]:
initial_regrets_eg = online_strategy_user_eg.run_initial_learning()

--Timestamp 1028504458 after 10000 iterations: sum of regrets 1037.125 top actions [(9, 'Film-Noir', 0.9459337723658056), (16, 'War', 0.8511904761904762), (5, 'Crime', 0.8386363636363637), (12, 'Mystery', 0.828893442622951), (13, 'Romance', 0.8230088495575222)]


In [316]:
print(initial_regrets_eg[:25])

[0.125, 0.125, 0.125, 1, 0.25, 0.375, 0.0, 1, 0.0, 0.0, 0.0, 0.375, 0.125, 0.0, 0.0, 0.375, 0.0, 1, 0.125, 0.375, 0.125, 0.125, 0.125, 0.125, 0.5]


As we can see, the epsilon-greedy strategy has found the same top-three genres, but the other two were determined to be the "Mystery" and "Romance", which have very close average rewards and are a bit less thematically aligned with the others. Also, here the sum of regrets is a bit smaller than for the UCB strategy, indicating that the epsilon-greedy strategy may focus on the nest genres earlier than the UCB due to its greedy nature. Let's see if the daily online learning over the next week changes anything:

In [317]:
online_regrets_eg = online_strategy_user_eg.run_online_learning(steps_ahead=7)

--Step 1 new_timestamp 1028505898 after 1000 iterations: sum of regrets 107.5 top actions [(9, 'Film-Noir', 0.9455081209113468), (16, 'War', 0.8547008547008544), (5, 'Crime', 0.8408119658119658), (12, 'Mystery', 0.8327067669172934), (7, 'Drama', 0.8218562874251496)]
--Step 2 new_timestamp 1028507338 after 1000 iterations: sum of regrets 102.125 top actions [(9, 'Film-Noir', 0.9455774400330851), (16, 'War', 0.852822580645161), (12, 'Mystery', 0.8374125874125875), (5, 'Crime', 0.8369140625), (7, 'Drama', 0.8234463276836158)]
--Step 3 new_timestamp 1028508778 after 1000 iterations: sum of regrets 99.75 top actions [(9, 'Film-Noir', 0.9458798549479914), (16, 'War', 0.8559782608695647), (12, 'Mystery', 0.8363486842105263), (5, 'Crime', 0.8314814814814816), (7, 'Drama', 0.8284574468085106)]
--Step 4 new_timestamp 1028510218 after 1000 iterations: sum of regrets 105.375 top actions [(9, 'Film-Noir', 0.9458355472901162), (16, 'War', 0.8600993377483439), (7, 'Drama', 0.834777227722772), (5, 'Cr

Due to the very close average reward values for the genres in the positions 3-5, we have seen them changing places several times. "Drama", always present in the UCB strategy's top-5, has made it to the third position this week by a small margin, and the "Romance" is gone from the list of the top-5 best genres. This highlights how the strategy adapts to the newest developments in the user's movie choices, and at the same time has memory about the overall picture with the average reward for each action.

Also, here the sum of regrets is slightly higher than for the same period for the UCB strategy, indicating that it may be better at judging the overall picture through the higher level of exploitation on this stage. Epsilon-greedy strategy at the same time it stuck with the set level of exploration, and therefore may pick up more regret during such iterations of online learning.

At the same time, this strategy is definitely more update-sensitive than the UCB strategy, possibly thanks to the selected $\epsilon$ value of 0.2. This means that each 5th action is reserved for a random exploration, and it can evidently lead to some higher volatility in the results on each step of the online learning.

Next, let's review the **Thompson sampling strategy**.

In [334]:
online_strategy_user_ts = OnlineStrategyLearner(df_ratings,
                                             len(genres_list),
                                             genres_list,
                                             genres_movie_list,
                                             start_timestamp,
                                             24*60, # Update each day = 24*60 minutes
                                             10000,
                                             MultiArmedBandit,
                                             ThompsonSamplingStrategy,
                                             user_id=4169,
                                             max_number_rounds=10000,
                                             max_number_online_rounds=1000)

In [335]:
initial_regrets_ts = online_strategy_user_ts.run_initial_learning()

--Timestamp 1028504458 after 10000 iterations: sum of regrets 618.375 top actions [(9, 'Film-Noir', 8963.5, 509.5), (16, 'War', 78.625, 11.375), (7, 'Drama', 73.0, 12.0), (5, 'Crime', 46.25, 9.75), (12, 'Mystery', 45.25, 9.75)]


In [336]:
print(initial_regrets_ts[:25])

[0.0, 0.125, 0.125, 0.0, 0.0, 0.125, 0.125, 0.0, 0.125, 0.25, 0.125, 1, 0.0, 0.0, 0.5, 0.125, 1, 0.0, 0.125, 1, 0.25, 0.25, 0.375, 0.125, 0.125]


As we can see, the recommended genres here are very similar to the both previous strategies. However, the resulting sum of regrets is much smaller than in both, indicating that the Thompson sampling has converged to the best genres much faster, and started exploiting them. Let's see if the trend preserves for the dialy online learning for the following week:

In [337]:
online_regrets_ts = online_strategy_user_ts.run_online_learning(steps_ahead=7)

--Step 1 new_timestamp 1028505898 after 1000 iterations: sum of regrets 59.125 top actions [(9, 'Film-Noir', 9894.875, 565.125), (16, 'War', 83.875, 12.125), (7, 'Drama', 73.75, 12.25), (5, 'Crime', 48.125, 9.875), (12, 'Mystery', 46.0, 10.0)]
--Step 2 new_timestamp 1028507338 after 1000 iterations: sum of regrets 51.25 top actions [(9, 'Film-Noir', 10837.25, 615.75), (16, 'War', 87.5, 12.5), (7, 'Drama', 73.75, 12.25), (5, 'Crime', 49.0, 10.0), (12, 'Mystery', 47.0, 10.0)]
--Step 3 new_timestamp 1028508778 after 1000 iterations: sum of regrets 59.625 top actions [(9, 'Film-Noir', 11770.625, 673.375), (16, 'War', 90.25, 13.75), (7, 'Drama', 75.375, 12.625), (5, 'Crime', 49.0, 10.0), (12, 'Mystery', 48.625, 10.375)]
--Step 4 new_timestamp 1028510218 after 1000 iterations: sum of regrets 55.25 top actions [(9, 'Film-Noir', 12712.625, 726.375), (16, 'War', 92.25, 13.75), (7, 'Drama', 75.375, 12.625), (12, 'Mystery', 48.625, 10.375), (5, 'Crime', 49.0, 11.0)]
--Step 5 new_timestamp 1028511

As we can see, the Thompson sampling reaches almost the same values of the sum of regrets on the online learning iterations as the UCB method, while it also actively becomes more confident in the top predictions, which become reinforced by the new rounds for that genres. That is only happening on a smaller degree for the UCB due to the average reward values there having an upper boundary of 1. Therefore, for using Thompson sampling for the large periods of online learning it most likely would be better to introduce some sort of alpha and beta parameter normalizations.

The top-5 gebres here have remained the same, once again demonstrating the difference wit hthe epsilon-greedy strategy. This is the result of the Thompson sampling focusing more on the exploitation here due to reduced upper bounds and a lot of exploring, likely we need to increase the bounds periodically for the online learning on the long periods. Still, "Mystery" and "Crime" exchange places in the middle of the weak, indicating how the user's preferences influence the coefficients of the distributions from where the actions are drwan in this strategy.